## Imports core libraries for Spark ETL:

In [42]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, regexp_extract, trim, when

Creates a SparkSession named "Data Lake Builder" running locally.

In [43]:
spark = SparkSession.builder \
    .appName("Data Lake Builder") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.instances", "1") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "5g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()

In [44]:
RAW_ZONE = "/datalake/Raw"
Bronze_Zone = "/datalake/Bronze"
CLEANSED_ZONE = "/datalake/Silver"
GOLD_ZONE = "/datalake/Gold"

Loads CSV files from the RAW_ZONE into Spark DataFrames with headers and automatic schema inference for further processing.

In [45]:
accounts_raw = spark.read.csv(f"{RAW_ZONE}/accounts.csv", header=True, inferSchema=True)
details_raw = spark.read.csv(f"{RAW_ZONE}/account_details.csv", header=True, inferSchema=True)
Person_raw = spark.read.csv(f"{RAW_ZONE}/Person.csv", header=True, inferSchema=True)
profile_raw = spark.read.csv(f"{RAW_ZONE}/person_profile.csv", header=True, inferSchema=True)
iden_raw = spark.read.csv(f"{RAW_ZONE}/person_iden.csv", header=True, inferSchema=True)

26/04/24 00:25:51 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


### Write data in a Bronze layer 

In [46]:
accounts_raw.write.mode("overwrite").parquet(f"{Bronze_Zone}/accounts/")
details_raw.write.mode("overwrite").parquet(f"{Bronze_Zone}/account_details/")
Person_raw.write.mode("overwrite").parquet(f"{Bronze_Zone}/Person/")
profile_raw.write.mode("overwrite").parquet(f"{Bronze_Zone}/person_profile/")
iden_raw.write.mode("overwrite").parquet(f"{Bronze_Zone}/person_iden/")

### Cleansing Data

In [47]:
# Accounts
accounts_clean = (
    accounts_raw
    .toDF(*[c.strip() for c in accounts_raw.columns])
    .withColumn("Update_Status_Date", to_date(col("Date"), "dd-MMM-yy"))
    .withColumnRenamed("Acc no", "Acc_No")
    .drop("Date")
    .dropDuplicates()
)

# account_details
details_clean = (
    details_raw
    .toDF(*[c.strip() for c in details_raw.columns])
    .withColumn("Update_Type_Date", to_date(col("Date"), "dd-MMM-yy"))
    .withColumnRenamed("Acc no", "Acc_No")
    .drop("Date")
    .dropDuplicates()
)

# Person
person_clean = (
    Person_raw
    .toDF(*[c.strip() for c in Person_raw.columns])
    .withColumnRenamed("Acc no", "Acc_No")
    .dropDuplicates()
)

# person_profile
profile_clean = (
    profile_raw
    .toDF(*[c.strip() for c in profile_raw.columns])
    .withColumn("Update_Name_Date", to_date(col("Date"), "dd-MMM-yy"))
    .drop("Date")
    .dropDuplicates()
)

# person_iden
iden_clean = (
    iden_raw
    .toDF(*[c.strip() for c in iden_raw.columns])
    .withColumn("Update_Id_Date", to_date(col("Date"), "dd-MMM-yy"))
    .withColumn("id_number", trim(regexp_extract(col("Id"), r"^([A-Za-z0-9]+)", 1)))
    .withColumn("id_type", regexp_extract(col("Id"), r"\(([^)]+)\)", 1))
    .withColumn("id_type", when(col("id_type") == "", None).otherwise(col("id_type")))
    .drop("Date", "Id")
    .dropDuplicates()
)

In [48]:
accounts_clean.show()
details_clean.show()
person_clean.show()
profile_clean.show()
iden_clean.show()

+------+--------+------------------+
|Acc_No|  Status|Update_Status_Date|
+------+--------+------------------+
|   456|  Active|        2022-02-01|
|   123|  Active|        2022-01-01|
|   123|InActive|        2022-03-01|
+------+--------+------------------+



+------+----+----------------+
|Acc_No|type|Update_Type_Date|
+------+----+----------------+
|   123|  CC|      2022-03-01|
|   456|Loan|      2022-02-01|
|   123|  CC|      2022-01-01|
+------+----+----------------+

+------+------+
|Acc_No|Person|
+------+------+
|   123|     X|
|   456|     Y|
|   456|     Z|
+------+------+

+------+--------+----------------+
|Person|    Name|Update_Name_Date|
+------+--------+----------------+
|     X|   Ahmed|      2022-01-01|
|     Y|    Hana|      2022-02-01|
|     Z|    Rana|      2022-02-01|
|     Z|Rana Ali|      2022-03-01|
+------+--------+----------------+



+------+--------------+---------+-------+
|Person|Update_Id_Date|id_number|id_type|
+------+--------------+---------+-------+
|     Z|    2022-02-01|      ID3|    NID|
|     Z|    2022-04-01|      ID4|   PASS|
|     Y|    2022-02-01|      ID2|   NULL|
|     X|    2022-01-01|      ID1|   NULL|
+------+--------------+---------+-------+



### Write data in a silver layer 

In [49]:
accounts_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/accounts/")
details_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/account_details/")
person_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/Person/")
profile_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/person_profile/")
iden_clean.write.mode("overwrite").parquet(f"{CLEANSED_ZONE}/person_iden/")

Writes the Gold dimension table to Parquet format in the GOLD_ZONE with overwrite mode and partitions the data by Matching_date for optimized querying.

In [50]:
dim_account_gold = accounts_clean.join(
    details_clean, 
    (accounts_clean.Acc_No == details_clean.Acc_No) & 
    (accounts_clean.Update_Status_Date == details_clean.Update_Type_Date), 
    "left"
).select(
    accounts_clean["Acc_No"],
    details_clean["type"].alias("Account_Type"),
    accounts_clean["Status"].alias("Account_Status"),
    accounts_clean["Update_Status_Date"].alias("Matching_date")
)

In [51]:
dim_account_gold.write\
            .mode("overwrite")\
            .partitionBy("Matching_date")\
            .parquet(f"{GOLD_ZONE}/dim_account/")


Joins person profile with person data to build a Gold level person dimension table, then saves it as partitioned Parquet files in the GOLD_ZONE by Update_Name_Date.

In [52]:
dim_person_gold = profile_clean.join(
    person_clean,
    profile_clean.Person == person_clean.Person,
    "left"
).select(
    person_clean["Acc_no"],
    profile_clean["Person"],
    profile_clean["Name"],
    profile_clean["Update_Name_Date"]
)

In [53]:
dim_person_gold.write\
            .mode("overwrite")\
            .partitionBy("Update_Name_Date")\
            .parquet(f"{GOLD_ZONE}/dim_person/")


Writes the identity cleaned dataset to the GOLD_ZONE as partitioned Parquet files using Update_Id_Date for optimized storage and querying.

In [54]:
iden_clean.write\
            .mode("overwrite")\
            .partitionBy("Update_Id_Date")\
            .parquet(f"{GOLD_ZONE}/dim_Identity/")

In [55]:
# Stops the active Spark session and releases all associated resources.

spark.stop()

26/04/24 00:29:02 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-e6771e90-53c0-4073-bc7e-2c3ab54d6df6/pyspark-4ff43b64-dd22-40b0-b599-759aaa61adb2. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-e6771e90-53c0-4073-bc7e-2c3ab54d6df6/pyspark-4ff43b64-dd22-40b0-b599-759aaa61adb2
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun